# Actividad 3: Modelo de Lenguaje

### **Construye una Red Neuronal Recurrente o Transformer para una tarea de generación de texto.**

En esta actividad por grupos, teneis que construir una Red Recurrente Elman, LSTM, GRU, o un Transformer (a elegir), que entrenareis con un dataset de texto (también a elegir). Una vez entrenado, el modelo tiene que ser capaz de generar texto.

**Para aprobar, el código debe ejecutar sin errores importantes** (no pasa nada por algun warning leve de pytorch), la red se debe entrenar y la función de perdida disminuir. Por último, en la inferencia, **la red debe ser capaz de imprimir texto**.

**No hay ningún requisito de calidad del texto generado**. Modelar bien el lenguaje humano es una tarea excepcionalmente complicada y que requiere de mucha capacidad de procesamiento y una gran cantidad de datos. No espero que ninguno de los textos generados tenga sentido. Aún así, se valorará positivamente que el texto generado tenga un parecido razonable con tener algo de coherencia.

Este es un ejercicio más complicado que las actividades previas, pero es en grupo, y disponeis de más tiempo, semana y media. Aconsejo partir de los ejercicios resueltos, leerlos y entenderlos, y tocar poco a poco el código, para ver que efecto tiene. Si os atascais, no dudeis en publicar vuestras dudas en el tablon para que otros alumnos o el profesor os pueda ayudar.

**La entrega de la actividad será el archivo del cuaderno de Jupyter (archivo .ipynb). Puede descargarse desde Archivo > Descargar > Descargar .ipynb**. Si usais algun archivo para entrenar que tengais en vuestro PC, debeis entregar tambien ese archivo.

In [18]:
# importamos la libreria pytorch
import torch
from torch import nn
from torch import optim #las funciones de optimizacion (gradient descent)
from torch.optim.lr_scheduler import StepLR #learning rate decay
import torch.nn.functional as F #convencion
from torchvision import datasets, transforms
import re

# matplotlib para pintar graficas
import matplotlib.pyplot as plt
%matplotlib inline

# para poder descargar el dataset desde una URL
# o leerlo de tu PC. si el archivo es local,
# # por favor mandadlo junto con el cuaderno .ipynb
# # para que en la correccion pueda replicar vuestro entrenamiento
import requests
from io import open
import string # Import the string module for punctuation handling

In [19]:
# usamos la GPU si esta disponible
if torch.cuda.is_available():
  device = torch.device("cuda")
  print("Dispositivo usado: GPU con CUDA")
else:
  device = torch.device("cpu")
  print("Dispositivo usado: CPU")

Dispositivo usado: GPU con CUDA


In [22]:
# el diccionario es una estructura de datos
# que guarda la relación palabra <-> id de token
class Dictionary(object):
    def __init__(self):
        self.token2idx = {}
        self.idx2token = []

    # añadir un token al diccionario
    def add_token(self, token):
        # si un token no estaba en el diccionario
        # se añade su entrada
        if token not in self.token2idx:
            self.idx2token.append(token)
            self.token2idx[token] = len(self.idx2token) - 1
        return self.token2idx[token]

    # funcion que devuelve el tamaño del diccionario
    # (cuantos tokens distintos hay)
    def __len__(self):
        return len(self.idx2token)

    # imprimir la lista de tokens distintos
    def __repr__(self):
        string = ""
        for char in sorted(self.idx2token):
            string += char
        return string

class Corpus(object):
    def __init__(self):
        self.dictionary = Dictionary()

        # URLs con el texto (son archivos .txt)
        url_train = "https://gist.githubusercontent.com/guillepowermetal/b7d3e8135ba228ca6b5e195b15958ede/raw/c483891c428cc8872b9de0af45c9446d6e31699e/Odisea_train.txt"
        url_test = "https://gist.githubusercontent.com/guillepowermetal/b7d3e8135ba228ca6b5e195b15958ede/raw/c483891c428cc8872b9de0af45c9446d6e31699e/Odisea_test.txt"
        url_validation = "https://gist.githubusercontent.com/guillepowermetal/b7d3e8135ba228ca6b5e195b15958ede/raw/c483891c428cc8872b9de0af45c9446d6e31699e/Odisea_validate.txt"

        # se pasa el texto del .txt a la funcion tokenize()
        self.train = self.tokenize(requests.get(url_train).text)
        self.validation = self.tokenize(requests.get(url_validation).text)
        self.test = self.tokenize(requests.get(url_test).text)

        print("conversion a tokens completada!")
        print(len(self.dictionary), "tokens distintos en el diccionario:")
        print(self.dictionary)
        print(self.train.shape[0], "tokens en el split de train")

    # tokeniza el texto
    def tokenize(self, text):

        # lista donde se va a guardar la secuencia de texto convertida en tokens
        tokenseq = []

        # añadir los tokens al diccionario (por si todavia no estan)
        # modelo de lenguaje a nivel de caracteres = los tokens son caracteres
        for line in text:
            for char in line:
                self.dictionary.add_token(char) # se añade cada caracter al diccionario

        # convertimos a tokens cada caracter del texto
        for line in text:
            # ids = []
            for char in line:
                tokenseq.append(torch.tensor(self.dictionary.token2idx[char]).type(torch.int64))

        # se convierte tokenseq a un tensor de pytorch
        embed = torch.tensor(tokenseq)
        return embed

block_size = 32 # tamaño de contexto, lo que mira hacia atras el modelo para hacer una prediccion
batch_size = 16 # batches, cuantas secuencias de texto se procesan a la vez

# genera un par de datos de entrada y targets correspondientes
# el target es el texto desplazado un caracter
def get_batch(source):
    data = source
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# Tokenización

Se modifico la clase Corpus para tokenizar por palabra en vez de por caracter

In [35]:
# el diccionario es una estructura de datos
# que guarda la relación palabra <-> id de token
class Dictionary(object):
    def __init__(self):
        self.token2idx = {}
        self.idx2token = []

    # añadir un token al diccionario
    def add_token(self, token):
        # si un token no estaba en el diccionario
        # se añade su entrada
        if token not in self.token2idx:
            self.idx2token.append(token)
            self.token2idx[token] = len(self.idx2token) - 1
        return self.token2idx[token]

    # funcion que devuelve el tamaño del diccionario
    # (cuantos tokens distintos hay)
    def __len__(self):
        return len(self.idx2token)

    # imprimir la lista de tokens distintos
    def __repr__(self):
        # Tokenizacion por palabra
        if len(self.idx2token) > 20: # mostrar un sample si el diccionario es muy grande
            return f"Dictionary with {len(self.idx2token)} tokens. Sample: {self.idx2token[:5]} ... {self.idx2token[-5:]}"
        else:
            return f"Dictionary with {len(self.idx2token)} tokens: {self.idx2token}"

class Corpus(object):
    def __init__(self):
        self.dictionary = Dictionary()
        # Add <unk> token primero para palabras out-of-vocabulary
        self.dictionary.add_token('<unk>')

        # URLs con el texto (son archivos .txt)
        # url_train = "https://gist.githubusercontent.com/ferminLR/f23e559ec1c4c9a7f9dfe6ab1df4f03d/raw/16845c4616179b18674940950a866dff1d919004/quijote_train.txt"
        # url_test = "https://gist.githubusercontent.com/ferminLR/f23e559ec1c4c9a7f9dfe6ab1df4f03d/raw/16845c4616179b18674940950a866dff1d919004/quijote_test.txt"
        # url_validation = "https://gist.githubusercontent.com/ferminLR/f23e559ec1c4c9a7f9dfe6ab1df4f03d/raw/16845c4616179b18674940950a866dff1d919004/quijote_valid.txt"

        # Dataset de la Odisea
        url_train = "https://gist.githubusercontent.com/guillepowermetal/b7d3e8135ba228ca6b5e195b15958ede/raw/c483891c428cc8872b9de0af45c9446d6e31699e/Odisea_train.txt"
        url_test = "https://gist.githubusercontent.com/guillepowermetal/b7d3e8135ba228ca6b5e195b15958ede/raw/c483891c428cc8872b9de0af45c9446d6e31699e/Odisea_test.txt"
        url_validation = "https://gist.githubusercontent.com/guillepowermetal/b7d3e8135ba228ca6b5e195b15958ede/raw/c483891c428cc8872b9de0af45c9446d6e31699e/Odisea_validate.txt"

        # Carga de datos raw
        raw_train_text = requests.get(url_train).text
        raw_validation_text = requests.get(url_validation).text
        raw_test_text = requests.get(url_test).text

        # Primer pass: construir el diccionario de todo el texto disponible para segurar que
        # todas las palabras sean mapeadas
        full_text_for_dict = raw_train_text + raw_validation_text + raw_test_text
        words_for_dict = self._split_text_into_words(full_text_for_dict)

        for word in words_for_dict:
            self.dictionary.add_token(word)

        # Tokenizar cada split en sequencias de word IDs
        self.train = self._tokenize_words_to_ids(raw_train_text)
        self.validation = self._tokenize_words_to_ids(raw_validation_text)
        self.test = self._tokenize_words_to_ids(raw_test_text)

        print("conversion a tokens completada!")
        print(len(self.dictionary), "tokens distintos en el diccionario:")
        print(self.dictionary)
        print(self.train.shape[0], "tokens en el split de train")

    # Helper para separar el texto en palabras usando regex
    def _split_text_into_words(self, text):
        # Convertir a minusculas
        text = text.lower()
        # Este regex captura sequencias de caracteres alphanumeric (palabras),
        # o sequencias de non-alphanumeric, non-whitespace characters (puntuaciones),
        # o newline (linea nueva). De esta manera se asegura que se creen tokens separads para puntuaciones.
        words = re.findall(r'\b\w+\b|[^\s\w]+|\n', text)
        # Filtrado de espacios si existen multiples espacios en blanco
        return [word for word in words if word.strip()]

    # Modified tokenization method to handle words
    def _tokenize_words_to_ids(self, text):
        tokenseq = []
        words = self._split_text_into_words(text)

        for word in words:
            # Get the index of the word, if not found, use the <unk> token's index
            token_id = self.dictionary.token2idx.get(word, self.dictionary.token2idx['<unk>'])
            tokenseq.append(torch.tensor(token_id).type(torch.int64))

        # se convierte tokenseq a un tensor de pytorch
        embed = torch.tensor(tokenseq)
        return embed

block_size = 32 # tamaño de contexto, lo que mira hacia atras el modelo para hacer una prediccion
batch_size = 16 # batches, cuantas secuencias de texto se procesan a la vez

# genera un par de datos de entrada y targets correspondientes
# el target es el texto desplazado un caracter
def get_batch(source):
    data = source
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [24]:
# carga de datos y generacion de diccionario
corpus = Corpus()
ntokens = len(corpus.dictionary)

# convertir los datos en batches
train_data = corpus.train
validation_data = corpus.validation
test_data = corpus.test

conversion a tokens completada!
12381 tokens distintos en el diccionario:
Dictionary with 12381 tokens. Sample: ['<unk>', 'odisea', 'canto', 'i', ':'] ... ['encogiéndose', 'perseguirlos', 'termine', 'enoje', 'juraran']
136345 tokens en el split de train


In [25]:
# Hiperparámetros de la red
# reducir n_embd, n_layer y epochs si no se usa GPU
# en CPU tardaría mucho el entrenamiento
# n_head tiene que ser divisor de n_embd

n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

# una cabeza de atencion
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape

        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)

        # calcula la puntuacion de atencion
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)

        # mascara para no atender al futuro
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)

        # salida del operador de atencion
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)

        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()

        # lista con las cabezas de atencion
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # el resultado de las cabezas se concatena a la salida
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

# bloque de un transformer
# atencion + mlp + sus residual connections
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()

        # el tamaño de cada cabeza de atencion es el
        # tamaño del embedding dividido por el tamaño de cada cabeza
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)

        # MLP
        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

        # layer normalization
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # suma -> conexion residual
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


vocab_size = ntokens

class Transformer(nn.Module):

    def __init__(self):
        super().__init__()

        # input embedding
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        # positional encoding
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # transformer blocks (decoders)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])

        # layer normalization
        self.ln_f = nn.LayerNorm(n_embd)

        # mlp al final
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, input, targets=None):
        B, T = input.shape

        # input embedding + positional encoding
        tok_emb = self.token_embedding_table(input)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb

        # decoder blocks
        x = self.blocks(x)

        # layer normalization final
        x = self.ln_f(x)

        # mlp al final
        logits = self.lm_head(x)

        # si no hay targets (inferencia), no hay loss
        if targets is None:
            loss = None
        else:
            # calculo de la perdida
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

In [26]:
# inferencia
def generate_text():
    # configurar el modelo en modo evaluacion o inferencia
    # se necesita para que capas como batchnorm o dropout se comporten correctamente
    model.eval()

    # generamos un input aleatorio
    input_token_id = torch.randint(ntokens, (1, 1), dtype=torch.long).to(device)
    input_sequence = input_token_id

    generated_tokens = []
    # Obtenemos el primer token y lo añadimos a la lista
    first_token = corpus.dictionary.idx2token[input_token_id.item()]
    generated_tokens.append(first_token)

    with torch.no_grad():
        # escribimos 'chars' tokens
        for i in range(chars - 1): # Generate chars-1 more tokens, as one is already generated

            # se ejecuta el modelo
            # Solo pasamos la parte relevante de la secuencia (últimos block_size tokens)
            input_block = input_sequence[:, -block_size:]
            output, _ = model(input_block)
            output = output[:,-1,:] # Obtenemos la predicción para el último token
            probs = F.softmax(output, dim=-1)

            # se saca el siguiente token a partir de la distribución de probabilidades
            input_next_id = torch.multinomial(probs, num_samples=1)
            input_sequence = torch.cat([input_sequence, input_next_id], dim=1)

            next_token = corpus.dictionary.idx2token[input_next_id.item()]
            generated_tokens.append(next_token)

    # Formatear y imprimir el texto generado para que sea legible
    final_text_raw = ""
    if generated_tokens:
        final_text_raw += generated_tokens[0]

    for i in range(1, len(generated_tokens)):
        token = generated_tokens[i]
        previous_token = generated_tokens[i-1]

        if token == "\n":
            final_text_raw += "\n"
        elif token in string.punctuation:
            # Si el token actual es puntuación, y el carácter anterior es un espacio,
            # eliminamos ese espacio antes de añadir la puntuación.
            if final_text_raw and final_text_raw[-1] == ' ':
                final_text_raw = final_text_raw[:-1]
            final_text_raw += token
        else: # El token actual es una palabra
            # Si el token anterior no era un salto de línea, añadimos un espacio.
            if previous_token != "\n":
                final_text_raw += " "
            final_text_raw += token

    # Añadir saltos de línea cada 15 palabras para mejor legibilidad
    words = final_text_raw.split(' ')
    formatted_text = []
    current_line_word_count = 0
    for word in words:
        if "\n" in word: # Handle explicit newlines
            parts = word.split("\n")
            for j, part in enumerate(parts):
                if part:
                    formatted_text.append(part)
                if j < len(parts) - 1:
                    formatted_text.append("\n")
                current_line_word_count = 0 # Reset word count after a newline
        else:
            formatted_text.append(word)
            current_line_word_count += 1

        if current_line_word_count >= 15:
            formatted_text.append("\n")
            current_line_word_count = 0

    print(' '.join(formatted_text).replace(' \n ', '\n').replace('\n ', '\n'))

In [27]:
# Como se comporta la red neuronal antes de entrenar?

# semilla del generador de numeros aleatorios
# para conseguir resultados deterministas
torch.manual_seed(42)

# inicializamos el modelo
model = Transformer()
model.to(device)

# configuramos como de largo queremos que sea el texto
chars = 2000

generate_text()

obligadas angostas cuyos describióle discretos relaja darles ofrendas oyendo aniquilarme esclavas irreprensibles lavaderos intenta honrarle
exhale morada destruyo mira sutil reparase peces telares comamos siniestros adaptaba profundidades atar nervios comprendiéndolo
ínclita domarlo arrancarle enorme mitad erigido venida lugar empleen oirá batían gratas pena verano poseído
recorría siga señalaba vigila ardía autólico acerquen soplaron lumbre libertar recreándose echaron pramnio mezclados unido
amenos acuesta ingenioso) ásperas perturbándole profundo corriendo diéronselo hablaban esas fenicios oto derramaban manifestarnos estén
partiera deidad quedáramos batallas adormecisteis echarlo erráticas comparecieron ordenándonos notes refugio poniendo rotonda noble retorna
audacia huerto tallado reconoció lavarle comían corriera diesen gruñido contemplábamos maquina haberse recuerdas antigua tablones
pagado respetado malvada logra pidas todavía comunica recibiría diversa degollando decoroso and

In [28]:
# funcion de entrenamiento
def train():

    # configurar el modelo en modo entrenamiento
    # se necesita para que capas como batchnorm o dropout se comporten correctamente
    model.train()

    # recorremos de manera aleatoria el set de datos
    # hacemos train_iters iteraciones
    for i in range(train_iters):

        # la funcion get_batch devuelve un pedazo de texto (el dato de entrada)
        # y otro pedazo desplazado un caracter (el target para ese dato de entrada)
        data, targets = get_batch(train_data)

        # calculamos la salida del modelo (prediccion)
        output, loss = model(data, targets)
        output = output.view(-1, ntokens)

        # se resetean a cero los gradientes
        optimizer.zero_grad(set_to_none=True)

        # se ejecuta el backpropagation para calcular el gradiente
        loss.backward()

        # se guarda el historico de la perdida para graficarlo
        loss_training.append(loss.item())

        # se actualizan los parametros de la red
        optimizer.step()

        # cada 500 iteraciones, imprimimos por pantalla la perdida
        print_interval = 500
        if i % print_interval == 0:
            print('   Loss: ', loss.item())

    print('Training Epoch completa\n')

In [29]:
# funcion de test
def test(split):

    # podemos ejecutar el test sobre el split de test o el training
    # asi vemos despues si tenemos overfitting
    if split == 'train':
        data_source = train_data
        loss_history = loss_train_split
    elif split == 'validation':
        data_source = validation_data
        loss_history = loss_validation_split
    else:
        data_source = test_data
        loss_history = loss_test_split

    # configurar el modelo en modo evaluacion o inferencia
    # se necesita para que capas como batchnorm o dropout se comporten correctamente
    model.eval()

    # variable para calcular la media de la perdida
    test_loss = 0

    # durante el test solo hacemos el forward pass y no necesitamos los gradientes
    with torch.no_grad():
        # recorremos de manera aleatoria el set de datos
        # hacemos eval_iters iteraciones
        for i in range(eval_iters):

            # la funcion get_batch devuelve un pedazo de texto (el dato de entrada)
            # y otro pedazo desplazado un caracter (el target para ese dato de entrada)
            data, targets = get_batch(data_source)

            # calculamos la salida del modelo (prediccion)
            output, loss = model(data, targets)
            output = output.view(-1, ntokens)

            # suma acumulada de la pérdida
            test_loss += loss

    # se divide por el numero de iteraciones para sacar la media de la perdida
    test_loss /= eval_iters

    # se guarda el historico de la perdida en el test para graficarlo
    loss_history.append(test_loss)

    # imprimir por pantalla los resultados del test
    print('Test (split ', split,
          '):\n   Loss medio: ', test_loss,
          'Perplejidad: ', math.exp(test_loss), '%\n')

    return test_loss

In [30]:
import math

# ejecutamos un test antes de empezar el entrenamiento
model = Transformer()
model.to(device)
loss_validation_split = []
criterion = nn.NLLLoss()
eval_iters = 200
test('validation')

Test (split  validation ):
   Loss medio:  tensor(9.5755, device='cuda:0') Perplejidad:  14407.405543982939 %



tensor(9.5755, device='cuda:0')

In [31]:
# entrenamos la red

# semilla del generador de numeros aleatorios
# para conseguir resultados deterministas
torch.manual_seed(42)

# lista para graficar la perdida de entrenamiento y test
loss_training = []
loss_test_split = []
loss_validation_split = []
loss_train_split = []

# Crear una instancia del modelo y pasarla al dispositivo
model = Transformer()
model.to(device)

# optimizador Adam
lr = 1e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

# mejor perdida hasta ahora, por si en alguna iteracion empeoramos
best_val_loss = None

# cuantos pedazos de texto se usan para entrenar y test en cada epoch
train_iters = 5000
eval_iters = 200

epochs = 5
for epoch in range(0, epochs):
    print('Epoch:', epoch)
    train()
    val_loss = test('validation')

    # guardamos los parametros de la red en un archivo 'transformer_model.pt'
    # si val_loss es la mejor que hemos hasta ahora
    if not best_val_loss or val_loss < best_val_loss:
        with open('transformer_model.pt', 'wb') as f:
            torch.save(model, f)
        best_val_loss = val_loss

Epoch: 0
   Loss:  9.58540153503418
   Loss:  5.4606733322143555
   Loss:  5.088406562805176
   Loss:  4.904047012329102
   Loss:  4.087575912475586
   Loss:  4.2060465812683105
   Loss:  3.971905469894409
   Loss:  3.7189817428588867
   Loss:  3.479942560195923
   Loss:  3.4797656536102295
Training Epoch completa

Test (split  validation ):
   Loss medio:  tensor(3.1358, device='cuda:0') Perplejidad:  23.005989274428032 %

Epoch: 1
   Loss:  3.1028101444244385
   Loss:  2.7200236320495605
   Loss:  2.968611478805542
   Loss:  3.109221935272217
   Loss:  2.6260111331939697
   Loss:  2.3589048385620117
   Loss:  2.5757806301116943
   Loss:  2.407172679901123
   Loss:  2.34159517288208
   Loss:  2.004854202270508
Training Epoch completa

Test (split  validation ):
   Loss medio:  tensor(2.1102, device='cuda:0') Perplejidad:  8.250148177691143 %

Epoch: 2
   Loss:  2.049135446548462
   Loss:  2.077138900756836
   Loss:  2.027040958404541
   Loss:  1.8968324661254883
   Loss:  1.9547796249

In [32]:
# calculamos la perdida sobre el set de test

# cargamos los parametros del modelo de 'transformer_model.pt'
with open('transformer_model.pt', 'rb') as f:
    model = torch.load(f, weights_only=False)

# Run on test data.
test_loss = test('test')

Test (split  test ):
   Loss medio:  tensor(1.1723, device='cuda:0') Perplejidad:  3.229254622988322 %



In [33]:
# semilla del generador de numeros aleatorios
# para conseguir resultados deterministas
torch.manual_seed(42)

# configuramos como de largo queremos que sea el texto
chars = 2000

# cargamos los parametros del modelo de 'transformer_model.pt'
with open('transformer_model.pt', 'rb') as f:
    model = torch.load(f, map_location=device, weights_only=False)

generate_text()

casaros aurora de corceles, a cada cual a gran puede orto? ¿ a amigo de
lo que te voy a pedir bronce. dime en qué manera, se te diría de
llegar allá el camino, y deja si todo lo quieren llevar a ningún lugar, como
la excelsa ciudad de alcínoo que las dejaron a ella hasta a un rey. apenas
el paciente divinal odiseo, a todos, ordenó cuyo palacio todos juntos, y quiso explorar la
voluntad de zeus prudente debilidad lo largo de zeus, pues administraba trae el haga que
en el camino, llevándola a un solo, donde después que era hijo de trabajo, ha
llegado por egipto, lo que has vuelto, desde que mal de resistir el mar que
la hueca emboscada previamente dispuesta. cosas excelentes son, te dañaron en el ánimo más pensamiento
de tu regreso sin hueca emboscada hombres, para que en el palacio de aquel peligro
las rodillas; y los cadáveres de obedecieron el arco, y ningún otro mortal, aunque con
tal ímpetu aquellos fuera de manejar vuelve de un varón muy irritado. al fin te
brazos con juramento que

# Mejoras
Con los parámetros originales vistos en clase la función `generate_text()` genera código, sin embargo no tiene ningún sentido ni significado. Para mejorar el entrenamiento y obtener texto que muestre estructura similar a oraciones humanas se modificaron los parámetros:
- `block_size` a 128
- `batch_size` a 32

Para tener mejor contexto a nivel palabra.
También se modificaron los parámeros:
- `n_embd` a 128
- `n_head` a 8
- `n_layer` a 6
- `dropout` de 0.1
para caputrar de mejor manera la dependencia de las palabras y evitar overfitting. to better capture word dependencies and prevent overfitting.

In [36]:
# Hiperparámetros ajustados para mejorar la relación de dependencia entre palabras

n_embd = 128
n_head = 8
n_layer = 6
dropout = 0.1

# una cabeza de atencion
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape

        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)

        # calcula la puntuacion de atencion
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)

        # mascara para no atender al futuro
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)

        # salida del operador de atencion
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)

        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()

        # lista con las cabezas de atencion
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # el resultado de las cabezas se concatena a la salida
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

# bloque de un transformer
# atencion + mlp + sus residual connections
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()

        # el tamaño de cada cabeza de atencion es el
        # tamaño del embedding dividido por el tamaño de cada cabeza
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)

        # MLP
        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

        # layer normalization
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # suma -> conexion residual
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


vocab_size = ntokens

class Transformer(nn.Module):

    def __init__(self):
        super().__init__()

        # input embedding
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        # positional encoding
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # transformer blocks (decoders)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])

        # layer normalization
        self.ln_f = nn.LayerNorm(n_embd)

        # mlp al final
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, input, targets=None):
        B, T = input.shape

        # input embedding + positional encoding
        tok_emb = self.token_embedding_table(input)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb

        # decoder blocks
        x = self.blocks(x)

        # layer normalization final
        x = self.ln_f(x)

        # mlp al final
        logits = self.lm_head(x)

        # si no hay targets (inferencia), no hay loss
        if targets is None:
            loss = None
        else:
            # calculo de la perdida
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


# Modificación de Batch

block_size = 128
batch_size = 32

In [37]:
block_size = 128 # tamaño de contexto, lo que mira hacia atras el modelo para hacer una prediccion
batch_size = 32 # batches, cuantas secuencias de texto se procesan a la vez

# genera un par de datos de entrada y targets correspondientes
# el target es el texto desplazado un caracter
def get_batch(source):
    data = source
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [38]:
import math

# entrenamos la red

# semilla del generador de numeros aleatorios
# para conseguir resultados deterministas
torch.manual_seed(42)

# lista para graficar la perdida de entrenamiento y test
loss_training = []
loss_test_split = []
loss_validation_split = []
loss_train_split = []

# Crear una instancia del modelo y pasarla al dispositivo
model = Transformer()
model.to(device)

# optimizador Adam
lr = 1e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

# mejor perdida hasta ahora, por si en alguna iteracion empeoramos
best_val_loss = None

# cuantos pedazos de texto se usan para entrenar y test en cada epoch
train_iters = 5000
eval_iters = 200

epochs = 5
for epoch in range(0, epochs):
    print('Epoch:', epoch)
    train()
    val_loss = test('validation')

    # guardamos los parametros de la red en un archivo 'transformer_model.pt'
    # si val_loss es la mejor que hemos hasta ahora
    if not best_val_loss or val_loss < best_val_loss:
        with open('transformer_model.pt', 'wb') as f:
            torch.save(model, f)
        best_val_loss = val_loss

Epoch: 0
   Loss:  9.580304145812988
   Loss:  4.430413246154785
   Loss:  3.3706326484680176
   Loss:  2.5185906887054443
   Loss:  2.1409239768981934
   Loss:  1.6882600784301758
   Loss:  1.42850923538208
   Loss:  1.129305362701416
   Loss:  0.9569735527038574
   Loss:  0.8834100961685181
Training Epoch completa

Test (split  validation ):
   Loss medio:  tensor(0.3816, device='cuda:0') Perplejidad:  1.464585687879062 %

Epoch: 1
   Loss:  0.8341154456138611
   Loss:  0.6785781979560852
   Loss:  0.6374713182449341
   Loss:  0.6135663986206055
   Loss:  0.5468235015869141
   Loss:  0.5079103112220764
   Loss:  0.475960373878479
   Loss:  0.4640684127807617
   Loss:  0.43821296095848083
   Loss:  0.3705941140651703
Training Epoch completa

Test (split  validation ):
   Loss medio:  tensor(0.1764, device='cuda:0') Perplejidad:  1.192949035193361 %

Epoch: 2
   Loss:  0.3723277449607849
   Loss:  0.35643208026885986
   Loss:  0.3725854754447937
   Loss:  0.3522467613220215
   Loss:  0

In [39]:
# calculamos la perdida sobre el set de test

# cargamos los parametros del modelo de 'transformer_model.pt'
with open('transformer_model.pt', 'rb') as f:
    model = torch.load(f, weights_only=False)

# Run on test data.
test_loss = test('test')

Test (split  test ):
   Loss medio:  tensor(0.1077, device='cuda:0') Perplejidad:  1.1137660884512826 %



In [40]:
torch.manual_seed(42)

# configuramos como de largo queremos que sea el texto
chars = 2000

# cargamos los parametros del modelo de 'transformer_model.pt'
with open('transformer_model.pt', 'rb') as f:
    model = torch.load(f, map_location=device, weights_only=False)

generate_text()

casaros a los pretendientes por esposa, le respondió de esta guisa: — ya, oh forastero,
el más infortunado de los huéspedes, y disfruta de lo que tienes adelante; pues la
divinidad te dará esto y te rehusará aquello, según le plegue a su ánimo puesto
que es todopoderosa. dijo, sacrificó las primicias a los sempiternos dioses y, libando el negro
vino, puso la copa en manos de odiseo, asolador de ciudades, que junto a su
porción estaba sentado. repartióles el pan mesaulio, a quien el porquerizo había adquirido por sí
solo, en la ausencia de su amo y sin ayuda de su dueña ni del
anciano laertes, comprándolo a unos tafios con sus propios bienes. todos metieron mano en las
viandas que tenían delante. y así que hubieron satisfecho el deseo de comer y de
beber, mesaulio quitó el pan, y ellos, hartos de pan y de carne, fuéronse sin
dilación a la cama. sobrevino una noche mala y sin luna, en la cual zeus
llovió sin cesar, y el lluvioso céfiro sopló continuamente y con gran furia. y odiseo
habló